# The largest point estimate is not a promotion decision

These are demonstration recommendations on a public dataset; no real reader's identity is present and no recommendation is personalized to a real person.

Decision date: 2026-09-27.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
e = json.loads((root / 'results/manifest.json').read_text(encoding='utf-8'))
print('Population:', e['dataset'])
print('Protocol:', e['protocol']['name'])


Population: {'name': 'goodbooks-10k complete catalog and ratings', 'backend': 'pinned public CSV -> parquet -> NumPy/SciPy exact scoring', 'source': 'https://github.com/zygmuntz/goodbooks-10k', 'source_commit': '6dd165b555a7b47b2dd36743a425776e641ff50c', 'license': 'CC BY-SA 4.0', 'source_rows': 5976479, 'catalog_size': 10000, 'interactions': 5976479, 'train_interactions': 4781183, 'test_interactions': 1195296, 'positive_training_interactions': 3286330, 'readers': 53424, 'evaluated_users': 5000}
Protocol: Global source-order holdout, full catalog


In [2]:
table = pd.DataFrame([{'model': r['label'], 'ndcg': r['ndcg']['mean'], 'low': r['ndcg']['low'], 'high': r['ndcg']['high']} for r in e['metrics']])
print(table.to_string(index=False))
print('Observed and corrected selection:', e['winner'])
print(pd.DataFrame(e['comparisons'])[['left', 'right', 'p', 'q', 'significant']].to_string(index=False))


           model     ndcg      low     high
      Popularity 0.040616 0.037258 0.044149
     Item cosine 0.054081 0.050718 0.057921
    Implicit ALS 0.054562 0.051263 0.057931
     Fixed blend 0.053771 0.050383 0.057598
  Content TF-IDF 0.041068 0.038087 0.044168
      LambdaMART 0.043963 0.041464 0.046612
Two-tower neural 0.001858 0.001399 0.002364
  Session cosine 0.089968 0.085519 0.094682
 Elman recurrent 0.023200 0.021198 0.025222
Observed and corrected selection: {'observed': 'session_cosine', 'corrected': 'session_cosine'}
          left          right      p        q  significant
    popularity    item_cosine 0.0002 0.000240         True
    popularity            als 0.0002 0.000240         True
    popularity          blend 0.0002 0.000240         True
    popularity        content 0.8416 0.841600        False
    popularity     lambdamart 0.1042 0.117225        False
    popularity      two_tower 0.0002 0.000240         True
    popularity session_cosine 0.0002 0.000240      

## Dead end retained: promote the initial blend because it sounds richer

The original bounded release's paired evidence did not establish the blend above ALS. This is an actual earlier measured result, preserved separately from the expanded run. Complexity and a large point estimate were rejected as sufficient promotion evidence.

In [3]:
history = json.loads((root / 'results/history/bounded-v0.1.0.json').read_text(encoding='utf-8'))
pair = [r for r in history['comparisons'] if {r['left'], r['right']} == {'als', 'blend'}]
print('Original bounded population:', history['dataset'])
print('Recorded ALS versus blend comparison:', pair)
assert pair and not pair[0]['significant']
for shortcut in ('split', 'sampled'):
    print(shortcut, pd.DataFrame([{'model':r['label'], 'reference':r['full']['mean'], 'shortcut':r['shortcut']['mean'], 'delta':r['delta']['mean']} for r in e['inflation'][shortcut]]).to_string(index=False))
print('Cold-reader investigation:', pd.DataFrame(e['cold_reader_analysis']['cohorts']).to_string(index=False))
print('Neural training:', e['neural_training']['two_tower'])
print('Measured geometry:', e['neural_training']['geometry'])
print('Local pgvector experiment:', e['pgvector'])
print('The neural loss fell, but held-out quality remained weak. Retain this failed architecture choice; do not infer global dead ReLU from a single query vector.')


Original bounded population: {'name': 'goodbooks-10k bounded subset', 'backend': 'pinned public CSV -> parquet -> NumPy/SciPy exact scoring', 'source': 'https://github.com/zygmuntz/goodbooks-10k', 'source_commit': '6dd165b555a7b47b2dd36743a425776e641ff50c', 'license': 'CC BY-SA 4.0', 'source_rows': 300000, 'catalog_size': 1000, 'interactions': 170264, 'train_interactions': 137192, 'test_interactions': 33072, 'positive_training_interactions': 92085, 'readers': 4630, 'evaluated_users': 500}
Recorded ALS versus blend comparison: [{'left': 'als', 'right': 'blend', 'difference': {'mean': 7.094407089772403e-05, 'low': -0.012193434588785171, 'high': 0.012937956877092762}, 'p': 0.9896, 'q': 0.9895999999999999, 'significant': False}]
split            model  reference  shortcut     delta
      Popularity   0.040616  0.090886  0.050269
     Item cosine   0.054081  0.270521  0.216440
    Implicit ALS   0.054562  0.190662  0.136100
     Fixed blend   0.053771  0.253859  0.200088
  Content TF-IDF   